# 01 · Data overview

Goal: understand the tables, their size and their quirks **before** answering any business question.

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect("../data/instacart.duckdb", read_only=True)

def q(sql):
    """Run a SQL query and return a pandas DataFrame."""
    return con.sql(sql).df()

q("SHOW TABLES")

,name
0,aisles
1,departments
2,order_products
3,order_products_prior
4,order_products_train
5,orders
6,products


## 1.1 Table sizes
How many rows does each table have?

In [2]:
q("""
SELECT 'orders' AS table_name, COUNT(*) AS n_rows FROM orders
UNION ALL
SELECT 'order_products_prior', COUNT(*) FROM order_products_prior
UNION ALL
SELECT 'products', COUNT(*) FROM products
UNION ALL
SELECT 'aisles', COUNT(*) FROM aisles
UNION ALL
SELECT 'departments', COUNT(*) FROM departments
UNION ALL
SELECT 'order_products_train', COUNT(*) FROM order_products_train
""")

,table_name,n_rows
0,orders,3421083
1,order_products_prior,32434489
2,products,49688
3,aisles,134
4,departments,21
5,order_products_train,1384617


## 1.2 Orders by `eval_set`
How many orders are in `prior`, `train` and `test`? Do `test` orders have products?

In [3]:
q("""
SELECT eval_set, count(*) as n_orders
FROM orders
GROUP BY eval_set
ORDER BY n_orders DESC
""")

,eval_set,n_orders
0,prior,3214874
1,train,131209
2,test,75000


In [4]:
q("""
SELECT o.eval_set,
       COUNT(DISTINCT o.order_id) AS n_orders,
       COUNT(op.product_id)       AS n_product_rows
FROM orders o
LEFT JOIN order_products op ON o.order_id = op.order_id
GROUP BY o.eval_set
""")

,eval_set,n_orders,n_product_rows
0,prior,3214874,32434489
1,test,75000,0
2,train,131209,1384617


Test orders (75,000) have no products: they can be used only for timing analysis (day/hour), not for basket analysis.

## 1.3 Users
How many unique users? How many orders per user (min, max, average, median)?

In [5]:
q("""
SELECT COUNT(DISTINCT user_id) as n_unique_users
FROM orders
""")

,n_unique_users
0,206209


In [6]:
q("""
SELECT MIN(n_orders)    AS min_orders,
       MAX(n_orders)    AS max_orders,
       AVG(n_orders)    AS avg_orders,
       MEDIAN(n_orders) AS median_orders
FROM (
    SELECT user_id, COUNT(*) AS n_orders
    FROM orders
    GROUP BY user_id
) AS orders_per_user
""")

,min_orders,max_orders,avg_orders,median_orders
0,4,100,16.590367,10.0


Users have between 4 and 100 orders: the dataset excludes customers with fewer than 4 orders (selection bias, loyalty metrics will be optimistic) and caps the heaviest users at 100.
Orders per user are right-skewed (mean 16.6 vs median 10), median = describe typical user.

## 1.4 Missing values
How many `NULL` values are in `days_since_prior_order`? Which orders are they?

In [7]:
q("""
SELECT COUNT(*) as n_null_days
FROM orders
WHERE days_since_prior_order IS NULL
""")

,n_null_days
0,206209


In [8]:
q("""
SELECT order_number, COUNT(*) as n_orders
FROM orders
WHERE days_since_prior_order IS NULL
GROUP BY order_number
""")


,order_number,n_orders
0,1,206209


days_since_prior_order is NULL only for each user's first order (206,209 rows = 1 per user). These are structural, not missing values: exclude them from interval analysis, do not fill with 0

## 1.5 Distribution of `days_since_prior_order`
Count orders for each value from 0 to 30. Is there anything strange at the end?

In [9]:
q("""
SELECT days_since_prior_order, COUNT(*) as n_orders
from orders
WHERE days_since_prior_order IS NOT NULL
GROUP BY days_since_prior_order
ORDER BY days_since_prior_order DESC
""")

,days_since_prior_order,n_orders
0,30.0,369323
1,29.0,19191
2,28.0,26777
3,27.0,22013
4,26.0,19016
5,25.0,19234
6,24.0,20712
7,23.0,23885
8,22.0,32012
9,21.0,45470


days_since_prior_order is capped at 30: the value 30 means "30 days or more" (11.5% of orders), not a monthly habit.
Strong weekly rhythm: clear peaks at 7, 14 and 21 days. 50% of reorders happen within 7 days, which fits a fixed weekly delivery model

## 1.6 Basket size
How many products are in an average order? (use `order_products_prior`)

In [10]:
q("""
SELECT MIN(n_products)    AS min_products,
       MAX(n_products)    AS max_products,
       AVG(n_products)    AS avg_products,
       MEDIAN(n_products) AS median_products
FROM (
    SELECT order_id, COUNT(*) AS n_products
    FROM order_products_prior
    GROUP BY order_id
) AS products_per_order
""")

,min_products,max_products,avg_products,median_products
0,1,145,10.088883,8.0


Typical basket: median 8 products (mean 10.1, max 145). Right-skewed: a few large stock-up orders raise the mean. Basket size drives delivery economics for a free-delivery model.